In [ ]:
import pandas as pd, numpy as np, datetime, joblib, time, warnings
import matplotlib.pyplot as plt, matplotlib.dates as mdates
from pathlib import Path
from openpyxl import load_workbook
from scipy.signal import savgol_filter
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
warnings.filterwarnings('ignore')
np.random.seed(42)

DATA_FILE  = Path(r'C:\Skripsi_WC\data\DATASET_FINAL_FLAGGED.xlsx')
DIR_MODELS = Path(r'C:\Skripsi_WC\output\models')
DIR_PLOTS  = Path(r'C:\Skripsi_WC\output\plots')
DIR_MODELS.mkdir(parents=True, exist_ok=True)
DIR_PLOTS.mkdir(parents=True, exist_ok=True)

if not DATA_FILE.exists():
    raise FileNotFoundError(f"File tidak ditemukan: {DATA_FILE}\nEdit path di atas.")

def parse_date(v):
    if isinstance(v, (datetime.datetime, datetime.date)): return pd.Timestamp(v)
    if isinstance(v, (int, float)) and not isinstance(v, bool):
        if 40000 < float(v) < 60000:
            return pd.Timestamp('1899-12-30') + pd.Timedelta(days=int(v))
    try: return pd.to_datetime(str(v), dayfirst=True, errors='coerce')
    except: return pd.NaT

def to_num(v):
    try: return float(str(v).replace(',', '.').strip())
    except: return np.nan

def read_sheet(wb, sname):
    ws = wb[sname]
    headers = list(ws.iter_rows(min_row=2, max_row=2, values_only=True))[0]
    rows = []
    for row in ws.iter_rows(min_row=4, values_only=True):
        if row[0] is None: continue
        r = {'_SHEET': sname}
        for i, col in enumerate(headers):
            if col is None: continue
            col = str(col).strip(); v = row[i] if i < len(row) else None
            if col == 'DATE':               r[col] = parse_date(v)
            elif col in ('WELL','LIFTING'): r[col] = str(v).strip() if v else None
            else:                           r[col] = to_num(v)
        rows.append(r)
    frame = pd.DataFrame(rows)
    return frame[frame['FIG BFPD'].notna() & (frame['FIG BFPD'] > 0)].copy()


print("Loading data historis...")
wb  = load_workbook(str(DATA_FILE), read_only=True, data_only=True)
dfs = [read_sheet(wb, s) for s in ['ESP','SRP','PCP','HPU']]
wb.close()
df = pd.concat(dfs, ignore_index=True)
df = df[df['DATE'].notna() & (df['DATE'].dt.year >= 2019)].copy()
df = df.sort_values(['WELL','DATE']).reset_index(drop=True)
print(f"  Loaded: {len(df):,} baris | {df['WELL'].nunique()} sumur")

df['DEPTH_ESP_NORM'] = df['PUMP_INTAKE_PROXY'] = df['DEPTH_SUBM_RATIO'] = 0.0
esp_mask  = df['LIFTING'] == 'ESP'
depth_max = df.loc[esp_mask, 'DEPTH ESP'].max() if 'DEPTH ESP' in df.columns else 1.0
if pd.notna(depth_max) and depth_max > 0:
    ev = esp_mask & df['DEPTH ESP'].notna() & (df['DEPTH ESP'] > 0)
    df.loc[ev, 'DEPTH_ESP_NORM'] = df.loc[ev,'DEPTH ESP'] / depth_max
    if 'SUBM' in df.columns:
        m  = ev & df['SUBM'].notna()
        fc = (df.loc[m,'DEPTH ESP'] - df.loc[m,'SUBM'].clip(lower=0)).clip(lower=0)
        df.loc[m, 'PUMP_INTAKE_PROXY'] = 0.4335 * 0.9 * (fc * 3.28084)
        m2 = ev & (df['SUBM'] > 0)
        df.loc[m2,'DEPTH_SUBM_RATIO'] = (
            df.loc[m2,'DEPTH ESP'] / df.loc[m2,'SUBM'].clip(lower=1)).clip(0,100)

pi_v = df['PI PROXY'].notna() & (df['PI PROXY'] > 0) if 'PI PROXY' in df.columns else pd.Series(False, index=df.index)
df['FLOW_PRODUCTIVITY_PROXY'] = np.nan
if pi_v.any():
    df.loc[pi_v,'FLOW_PRODUCTIVITY_PROXY'] = df.loc[pi_v,'FIG BFPD'] / df.loc[pi_v,'PI PROXY']
    cap = df['FLOW_PRODUCTIVITY_PROXY'].quantile(0.99)
    df['FLOW_PRODUCTIVITY_PROXY'] = df['FLOW_PRODUCTIVITY_PROXY'].clip(upper=cap)

for c in ['HZ','SPM','RPM','AMP']:
    df[f'HAS_{c}'] = df[c].notna().astype(int) if c in df.columns else 0
for c in ['FIG GAS','DT HRS','PROD EFF','FLP']:
    if c in df.columns: df[c] = df[c].fillna(0)

for fc in ['FLAG_WC_JUMP','FLAG_BFPD_SPIKE','FLAG_WC_ZERO']:
    if fc in df.columns: df = df[df[fc].fillna(0) == 0].copy()
df = df[(df['FIG %WC'] >= 0) & (df['FIG %WC'] <= 100)].copy()
if 'dP HEAD' in df.columns:
    df = df[df['dP HEAD'].notna() & (df['dP HEAD'] > -500) & (df['dP HEAD'] < 500)].copy()
ENC = {'SRP':0,'PCP':1,'ESP':2,'HPU':3,'NF':4}
df['LIFTING_CODE'] = df['LIFTING'].map(ENC).fillna(-1).astype(int)
df = df[df['LIFTING_CODE'] >= 0].copy()
df = df.sort_values(['WELL','DATE']).reset_index(drop=True)

TARGET = 'WC_SMOOTH'; df[TARGET] = np.nan
for well, grp in df.groupby('WELL'):
    arr = grp['FIG %WC'].values; n = len(arr)
    s   = pd.Series(arr).interpolate(limit_direction='both').ffill().bfill().fillna(50)
    a   = s.values.astype(float)
    if n >= 7:
        w = min(11, n if n%2==1 else n-1); w = max(w,5)
        try: a = np.clip(savgol_filter(a,w,2), 0, 100)
        except: pass
    df.loc[grp.index, TARGET] = a

for lag in [1,7,14,30]:
    df[f'WC_LAG{lag}'] = df.groupby('WELL')[TARGET].shift(lag)
for win in [7,14,30]:
    df[f'WC_ROLL{win}_MEAN'] = df.groupby('WELL')[TARGET].transform(
        lambda x: x.shift(1).rolling(win, min_periods=3).mean())
    df[f'WC_ROLL{win}_STD']  = df.groupby('WELL')[TARGET].transform(
        lambda x: x.shift(1).rolling(win, min_periods=3).std().fillna(0))
df['BFPD_CHANGE7'] = df.groupby('WELL')['FIG BFPD'].transform(lambda x: x.diff(7))
for col in ['HZ','SPM','RPM','AMP']:
    if col in df.columns: df[col] = df[col].fillna(0)

FEATURES = [
    'FIG BFPD','THP','CHP','SUBM','dP HEAD',
    'FLOW_PRODUCTIVITY_PROXY','PROD EFF',
    'DEPTH_ESP_NORM','PUMP_INTAKE_PROXY','DEPTH_SUBM_RATIO',
    'WC_LAG1','WC_LAG7','WC_LAG14','WC_LAG30',
    'WC_ROLL7_MEAN','WC_ROLL14_MEAN','WC_ROLL30_MEAN',
    'WC_ROLL7_STD','WC_ROLL30_STD','BFPD_CHANGE7',
    'LIFTING_CODE','SPM','RPM','HZ',
    'HAS_HZ','HAS_SPM','HAS_RPM','HAS_AMP',
]
FEATURES = [f for f in FEATURES if f in df.columns]

WAJIB = [TARGET,'FIG BFPD','THP','CHP','SUBM','dP HEAD','WC_LAG1','WC_LAG7']
WAJIB = [c for c in WAJIB if c in df.columns]
df_m = df.dropna(subset=WAJIB).copy()
df_s = df_m.sort_values('DATE').reset_index(drop=True)
cut  = int(len(df_s) * 0.80); cut_date = df_s.iloc[cut]['DATE']
df_train = df_s[df_s['DATE'] <  cut_date].copy()
df_test  = df_s[df_s['DATE'] >= cut_date].copy()
X_train = df_train[FEATURES]; y_train = df_train[TARGET]
X_test  = df_test[FEATURES];  y_test  = df_test[TARGET]

model = RandomForestRegressor(
    n_estimators=200, max_depth=None, min_samples_split=2,
    min_samples_leaf=1, max_features=0.4, n_jobs=-1, random_state=42)
print("Training model utama (tunggu ~15 detik)...")
t0 = time.time()
model.fit(X_train, y_train)
print(f"  ✓ Model utama selesai: {time.time()-t0:.0f} detik")

FEAT_NL = [f for f in FEATURES if 'WC_LAG' not in f and 'WC_ROLL' not in f]
rf_cs   = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=42)
print("Training cold-start model (tunggu ~5 detik)...")
t0 = time.time()
rf_cs.fit(X_train[FEAT_NL], y_train)
print(f"  ✓ Cold-start model selesai: {time.time()-t0:.0f} detik")

print()
print("=" * 55)
print("  ✓ SEMUA VARIABEL SIAP (standalone mode)")
print(f"  df          : {len(df):,} baris | {df['WELL'].nunique()} sumur")
print(f"  TARGET      : {TARGET}")
print(f"  FEATURES    : {len(FEATURES)} fitur")
print(f"  FEAT_NL     : {len(FEAT_NL)} fitur (cold-start)")
print(f"  model       : RandomForest (n=200, max_feat=0.4)")
print(f"  rf_cs       : RandomForest (n=100, cold-start)")
print(f"  depth_max   : {depth_max:.1f}")
print("=" * 55)

In [ ]:
diperlukan = {
    'df'        : 'DataFrame data historis',
    'TARGET'    : 'Nama kolom target (WC_SMOOTH)',
    'FEATURES'  : 'List 28 fitur model utama',
    'model'     : 'Model RF utama (sudah dilatih)',
    'rf_cs'     : 'Model RF cold-start (sudah dilatih)',
    'X_train'   : 'Training features',
    'y_train'   : 'Training target',
    'depth_max' : 'Nilai max DEPTH ESP untuk normalisasi',
}

semua_ok = True
for var, keterangan in diperlukan.items():
    ada = var in dir()
    status = "✓" if ada else "✗ TIDAK ADA"
    if not ada: semua_ok = False
    print(f"  {status}  {var:<12} — {keterangan}")

if not semua_ok:
    print()
    print("❌ Ada variabel yang hilang!")
    print("   → Jalankan Cell 0 di atas, ATAU")
    print("   → Jalankan notebook utama (Notebooks_Final_Kenali_Asam.ipynb) dulu")
else:
    # Bangun FEAT_NL jika belum ada (bisa saja tidak terdefinisi di scope ini)
    if 'FEAT_NL' not in dir():
        FEAT_NL = [f for f in FEATURES if 'WC_LAG' not in f and 'WC_ROLL' not in f]
        print(f"  ✓ FEAT_NL     — dibangun ulang: {len(FEAT_NL)} fitur")

    print()
    print("=" * 55)
    print("  SEMUA VARIABEL SIAP")
    print("=" * 55)
    print(f"  Data historis   : {len(df):,} baris | {df['WELL'].nunique()} sumur")
    print(f"  TARGET          : {TARGET}")
    print(f"  Fitur utama     : {len(FEATURES)}")
    print(f"  Fitur cold-start: {len(FEAT_NL)}")
    print()
    print(f"  19 fitur cold-start yang digunakan:")
    for i, f in enumerate(FEAT_NL, 1):
        print(f"    {i:>2}. {f}")

In [ ]:
SUMUR_BARU = [
    {
        'WELL'          : 'KAS-BARU-01',   # ganti dengan nama sumur
        'LIFTING'       : 'PCP',           # ESP / SRP / PCP / HPU
        'FIG BFPD'      : 120.0,           # bbl/hari
        'THP'           : 55.0,            # psi
        'CHP'           : 1.5,             # psi
        'SUBM'          : 200.0,           # ft
        'dP HEAD'       : -52.0,           # psi (-500 s/d 500)
        'RPM'           : 120.0,           # rpm PCP (0 jika bukan PCP)
        'HZ'            : 0.0,             # Hz ESP (0 jika bukan ESP)
        'SPM'           : 0.0,             # spm SRP (0 jika bukan SRP)
        'PROD EFF'      : 1.0,             # 0-1
        'PI PROXY'      : 1.5,             # isi 0 jika tidak diketahui
        'DEPTH ESP'     : 0.0,             # ft, isi hanya jika ESP
        'TANGGAL_MULAI' : '2026-06-01',    # YYYY-MM-DD
        'N_HARI'        : 30,              # berapa hari ke depan
    },

print(f"✓ {len(SUMUR_BARU)} sumur baru terdaftar:")
print()
for s in SUMUR_BARU:
    lt = s['LIFTING']
   
    warn = ""
    if lt == 'ESP': warn = "  ⚠️  PERINGATAN: R²(cold)=-0.06 untuk ESP — hasil tidak andal"
    if lt == 'HPU': warn = "  ⚠️  PERINGATAN: R²(cold)=-5.77 untuk HPU — hasil tidak andal"
    if lt == 'SRP': warn = "  ⚠️  Catatan: R²(cold)=0.13 untuk SRP — marginal"
    print(f"  {'✓' if lt=='PCP' else '⚠️' if lt in ['ESP','HPU'] else '~'}  "
          f"{s['WELL']:15s} ({lt}) | "
          f"BFPD={s['FIG BFPD']:.0f} bbl/d | "
          f"THP={s['THP']:.0f} psi | "
          f"dP={s['dP HEAD']:.1f} psi | "
          f"{s['TANGGAL_MULAI']} → +{s['N_HARI']} hari{warn}")

In [ ]:
ENC = {'SRP':0, 'PCP':1, 'ESP':2, 'HPU':3, 'NF':4}

rows_input = []

for sumur in SUMUR_BARU:
    well    = sumur['WELL']
    lifting = sumur['LIFTING']
    n_hari  = sumur['N_HARI']
    tgl_start = pd.Timestamp(sumur['TANGGAL_MULAI'])

    depth_esp = float(sumur.get('DEPTH ESP', 0) or 0)
    subm      = float(sumur.get('SUBM', 0) or 0)

    if lifting == 'ESP' and depth_esp > 0 and depth_max > 0:
        depth_esp_norm   = depth_esp / depth_max
        fluid_col        = max(0.0, depth_esp - max(0.0, subm))
        pump_intake      = 0.4335 * 0.9 * (fluid_col * 3.28084)
        depth_subm_ratio = min(100.0, depth_esp / max(1.0, subm))
    else:
        depth_esp_norm = pump_intake = depth_subm_ratio = 0.0

    pi_proxy = float(sumur.get('PI PROXY', 0) or 0)
    fpp      = min(670.0, sumur['FIG BFPD'] / pi_proxy) if pi_proxy > 0 else 0.0

    has_hz  = 1 if lifting == 'ESP' and float(sumur.get('HZ',  0) or 0) > 0 else 0
    has_spm = 1 if lifting == 'SRP' and float(sumur.get('SPM', 0) or 0) > 0 else 0
    has_rpm = 1 if lifting == 'PCP' and float(sumur.get('RPM', 0) or 0) > 0 else 0
    has_amp = 0  # sumur baru, belum ada data AMP

    lifting_code = ENC.get(lifting, -1)
    if lifting_code < 0:
        raise ValueError(f"LIFTING '{lifting}' tidak dikenal. Pilih: ESP, SRP, PCP, HPU")

    for i in range(n_hari):
        rows_input.append({
            'DATE'                   : tgl_start + pd.Timedelta(days=i),
            'WELL'                   : well,
            'LIFTING'                : lifting,
            'HARI_KE'                : i + 1,
            # 19 fitur cold-start
            'FIG BFPD'               : sumur['FIG BFPD'],
            'THP'                    : sumur['THP'],
            'CHP'                    : sumur['CHP'],
            'SUBM'                   : subm,
            'dP HEAD'                : sumur['dP HEAD'],
            'FLOW_PRODUCTIVITY_PROXY': fpp,
            'PROD EFF'               : float(sumur.get('PROD EFF', 1.0) or 1.0),
            'DEPTH_ESP_NORM'         : depth_esp_norm,
            'PUMP_INTAKE_PROXY'      : pump_intake,
            'DEPTH_SUBM_RATIO'       : depth_subm_ratio,
            'BFPD_CHANGE7'           : 0.0,   
            'LIFTING_CODE'           : lifting_code,
            'SPM'                    : float(sumur.get('SPM', 0) or 0),
            'RPM'                    : float(sumur.get('RPM', 0) or 0),
            'HZ'                     : float(sumur.get('HZ',  0) or 0),
            'HAS_HZ'                 : has_hz,
            'HAS_SPM'                : has_spm,
            'HAS_RPM'                : has_rpm,
            'HAS_AMP'                : has_amp,
        })

df_input = pd.DataFrame(rows_input)

for f in FEAT_NL:
    if f not in df_input.columns:
        df_input[f] = 0.0

print(f"✓ Input siap: {len(df_input):,} baris")
print(f"  ({df_input['WELL'].nunique()} sumur × N hari)")
print()
print("Preview baris pertama per sumur:")
preview_cols = ['DATE','WELL','LIFTING','FIG BFPD','THP','CHP',
                'dP HEAD','FLOW_PRODUCTIVITY_PROXY','LIFTING_CODE']
print(df_input.groupby('WELL').first().reset_index()[preview_cols].to_string(index=False))

In [ ]:
print("=" * 60)
print("  PREDIKSI COLD-START")
print("=" * 60)

X_pred = df_input[FEAT_NL].fillna(0)
wc_pred = rf_cs.predict(X_pred)
df_input['WC_PRED_%'] = np.clip(wc_pred, 0, 100).round(3)

 ── Uncertainty dari distribusi 100 pohon ────────────────────────────────
print("  Menghitung prediction interval dari distribusi pohon...")
tree_preds = np.column_stack([t.predict(X_pred) for t in rf_cs.estimators_])

df_input['WC_STD']    = np.std(tree_preds, axis=1).round(3)
df_input['WC_LO_80%'] = np.clip(np.percentile(tree_preds, 10,  axis=1), 0, 100).round(3)
df_input['WC_HI_80%'] = np.clip(np.percentile(tree_preds, 90,  axis=1), 0, 100).round(3)
df_input['WC_LO_95%'] = np.clip(np.percentile(tree_preds, 2.5, axis=1), 0, 100).round(3)
df_input['WC_HI_95%'] = np.clip(np.percentile(tree_preds, 97.5,axis=1), 0, 100).round(3)

print()
print(f"  {'WELL':<16} {'LT':<5} {'WC Min':>8} {'WC Mean':>9} {'WC Max':>8}"
      f" {'±Std':>7}  {'Keandalan'}")
print("  " + "─" * 72)

KEANDALAN = {
    'PCP': ('✓  Cukup andal',   'R²(cold)=0.41 | MAE~7%'),
    'SRP': ('~  Marginal',      'R²(cold)=0.13 | MAE~7%'),
    'ESP': ('⚠️  Tidak andal',   'R²(cold)=-0.06 — WC ESP ~konstan 99%'),
    'HPU': ('⚠️  Tidak andal',   'R²(cold)=-5.77 — sangat buruk'),
}

for well, grp in df_input.groupby('WELL'):
    lt    = grp['LIFTING'].iloc[0]
    wmin  = grp['WC_PRED_%'].min()
    wmean = grp['WC_PRED_%'].mean()
    wmax  = grp['WC_PRED_%'].max()
    std   = grp['WC_STD'].mean()
    label, detail = KEANDALAN.get(lt, ('?', ''))
    print(f"  {well:<16} {lt:<5} {wmin:>7.1f}% {wmean:>8.1f}% {wmax:>7.1f}%"
          f" {std:>6.2f}%  {label}")
    print(f"  {'':<16}  ↳ {detail}")

print()
print(f"  ✓ Total prediksi: {len(df_input):,} hari-sumur")

In [ ]:
LIFT_COLOR = {'ESP':'#F44336','SRP':'#2196F3','PCP':'#FF9800','HPU':'#4CAF50'}

wells = sorted(df_input['WELL'].unique())
ncols = min(2, len(wells))
nrows = (len(wells) + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(15, 5 * nrows))
axes_flat = np.array(axes).flat if nrows > 1 or ncols > 1 else [axes]

for ax, well in zip(axes_flat, wells):
    grp   = df_input[df_input['WELL']==well].sort_values('DATE')
    lt    = grp['LIFTING'].iloc[0]
    color = LIFT_COLOR.get(lt, '#666666')

    hist_lt = df[df['LIFTING'] == lt][TARGET].dropna()
    if len(hist_lt) > 100:
        p10 = hist_lt.quantile(0.10)
        p50 = hist_lt.quantile(0.50)
        p90 = hist_lt.quantile(0.90)
        ax.axhspan(p10, p90, alpha=0.06, color=color,
                   label=f'Rentang historis {lt} (P10–P90)')
        ax.axhline(p50, color=color, lw=0.8, ls=':', alpha=0.5,
                   label=f'Median historis {lt}: {p50:.1f}%')

    ax.fill_between(grp['DATE'], grp['WC_LO_95%'], grp['WC_HI_95%'],
                    color=color, alpha=0.10, label='95% PI')
    ax.fill_between(grp['DATE'], grp['WC_LO_80%'], grp['WC_HI_80%'],
                    color=color, alpha=0.20, label='80% PI')
    ax.plot(grp['DATE'], grp['WC_PRED_%'],
            color=color, lw=2.5, zorder=5, label='WC Prediksi (cold-start)')

    wc_mean = grp['WC_PRED_%'].mean()
    wc_std  = grp['WC_STD'].mean()
    label_lt, detail = KEANDALAN.get(lt, ('', ''))

    title_color = 'red' if lt in ['ESP','HPU'] else                   'darkorange' if lt == 'SRP' else 'black'
    ax.set_title(
        f"{well} ({lt}) | WC prediksi = {wc_mean:.1f}% ± {wc_std:.1f}%
"
        f"{label_lt}  {detail}",
        fontsize=9, fontweight='bold', color=title_color)

    ax.set_ylabel('Water Cut (%)', fontsize=9)
    ax.set_xlabel('Tanggal', fontsize=9)
    ax.set_ylim(max(0,  grp['WC_LO_95%'].min() - 8),
                min(100, grp['WC_HI_95%'].max() + 8))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right', fontsize=7.5)
    ax.legend(fontsize=7.5, loc='best', framealpha=0.85, ncol=1)

for ax in list(axes_flat)[len(wells):]:
    ax.set_visible(False)

plt.suptitle(
    "Prediksi Water Cut — Cold-Start (Sumur Baru Tanpa Riwayat WC)
"
    "Shading = rentang WC historis sumur sejenis | MAE referensi ~7% untuk PCP/SRP",
    fontsize=11, fontweight='bold', y=1.02)
plt.tight_layout()

fname = DIR_PLOTS / 'Fig_ColdStart_WC.png'
plt.savefig(str(fname), dpi=180, bbox_inches='tight', facecolor='white')
plt.show()
print(f"✓ Grafik tersimpan: {fname}")

In [ ]:
excel_path = DIR_MODELS / 'ColdStart_WC_SumurBaru.xlsx'

df_out = df_input.copy()
df_out['DATE'] = df_out['DATE'].dt.date

def get_label(lt):
    return {
        'PCP': 'Cukup andal — R²(cold)=0.41, MAE~7%',
        'SRP': 'Marginal — R²(cold)=0.13, MAE~7%',
        'ESP': 'TIDAK ANDAL — R²(cold)=-0.06, jangan digunakan',
        'HPU': 'TIDAK ANDAL — R²(cold)=-5.77, jangan digunakan',
    }.get(lt, 'Tidak diketahui')

df_out['KEANDALAN'] = df_out['LIFTING'].apply(get_label)

ringkasan = []
for well, grp in df_out.groupby('WELL'):
    lt = grp['LIFTING'].iloc[0]
    ringkasan.append({
        'WELL'              : well,
        'LIFTING'           : lt,
        'TANGGAL_MULAI'     : grp['DATE'].min(),
        'TANGGAL_AKHIR'     : grp['DATE'].max(),
        'N_HARI'            : len(grp),
        'WC_PRED_MEAN_%'    : round(grp['WC_PRED_%'].mean(), 2),
        'WC_PRED_MIN_%'     : round(grp['WC_PRED_%'].min(),  2),
        'WC_PRED_MAX_%'     : round(grp['WC_PRED_%'].max(),  2),
        'WC_STD_MEAN'       : round(grp['WC_STD'].mean(),    3),
        'PI_LO_80%'         : round(grp['WC_LO_80%'].mean(), 2),
        'PI_HI_80%'         : round(grp['WC_HI_80%'].mean(), 2),
        'R2_COLDSTART_REF'  : {'ESP':-0.0647,'SRP':0.1304,
                               'PCP':0.4135,'HPU':-5.7704}.get(lt, np.nan),
        'MAE_REF_%'         : {'ESP': '~tinggi','SRP':'~7%',
                               'PCP':'~7%','HPU':'~tinggi'}.get(lt, ''),
        'KEANDALAN'         : get_label(lt),
    })
df_ringkasan = pd.DataFrame(ringkasan)

out_cols = ['DATE','WELL','LIFTING','HARI_KE',
            'FIG BFPD','THP','CHP','SUBM','dP HEAD',
            'WC_PRED_%','WC_STD','WC_LO_80%','WC_HI_80%',
            'WC_LO_95%','WC_HI_95%','KEANDALAN']
out_cols = [c for c in out_cols if c in df_out.columns]

with pd.ExcelWriter(str(excel_path), engine='openpyxl') as writer:
    df_ringkasan.to_excel(writer, sheet_name='Ringkasan', index=False)
    df_out[out_cols].to_excel(writer, sheet_name='Detail_Semua', index=False)
    for lt in ['PCP','SRP','ESP','HPU']:
        sub = df_out[df_out['LIFTING']==lt]
        if len(sub) > 0:
            sub[out_cols].to_excel(writer, sheet_name=f'Detail_{lt}', index=False)

print(f"✓ Excel tersimpan: {excel_path}")
print()
print(f"  Sheet 'Ringkasan'    : {len(df_ringkasan)} sumur")
print(f"  Sheet 'Detail_Semua' : {len(df_out):,} baris")
for lt in ['PCP','SRP','ESP','HPU']:
    n = len(df_out[df_out['LIFTING']==lt])
    if n > 0: print(f"  Sheet 'Detail_{lt}'  : {n:,} baris")